# Krisha.kz - Сбор данных о продаже квартир
Парсинг объявлений о продаже квартир с крупнейшей платформы недвижимости Казахстана.

**Stack:** Python · requests · BeautifulSoup · pandas  
**Dataset:** 7,000 объявлений со всех регионов Казахстана

## 1. Настройка и тест на одной странице
Проверяем логику парсинга на первой странице перед полным запуском.

In [2]:
import requests
from bs4 import BeautifulSoup
url = "https://krisha.kz/prodazha/kvartiry/"
headers = {"User-agent": "Mozilla/5.0"}
response = requests.get(url, headers = headers)
soup = BeautifulSoup(response.content, 'html.parser')

list_of_links = []

cards = soup.findAll("div", attrs = {'class': 'a-card__inc'})
for card in cards:
    a_tag = card.find("a")
    if a_tag and a_tag.get("href"):
        link = "https://krisha.kz" + a_tag["href"]
        list_of_links.append(link)
print(list_of_links)

['https://krisha.kz/a/show/1012178700', 'https://krisha.kz/a/show/698590467', 'https://krisha.kz/a/show/694170572', 'https://krisha.kz/a/show/1012268913', 'https://krisha.kz/a/show/1010804699', 'https://krisha.kz/a/show/697546174', 'https://krisha.kz/a/show/691064914', 'https://krisha.kz/a/show/1011375260', 'https://krisha.kz/a/show/1007113061', 'https://krisha.kz/a/show/688070600', 'https://krisha.kz/a/show/1011768173', 'https://krisha.kz/a/show/1011619627', 'https://krisha.kz/a/show/1011582608', 'https://krisha.kz/a/show/1011485943', 'https://krisha.kz/a/show/1011047608', 'https://krisha.kz/a/show/1010808248', 'https://krisha.kz/a/show/1010704207', 'https://krisha.kz/a/show/1010375317', 'https://krisha.kz/a/show/1009011409', 'https://krisha.kz/a/show/1008955050', 'https://krisha.kz/a/show/1007548684', 'https://krisha.kz/a/show/1006368202', 'https://krisha.kz/a/show/1006042603']


## 2. Сбор ссылок со всех страниц (900 страниц)
Обходим все страницы листинга со случайными задержками (2–5 сек) для защиты от блокировки. Собрано ~19,000 ссылок, после дедупликации осталось **17,375 уникальных объявлений**.

## 3. Основной парсер - 7,000 объявлений
Парсим каждую страницу объявления: цена, количество комнат, площадь, этаж, район, тип дома, год постройки и 10+ дополнительных атрибутов из `dl` блоков. Реализован auto-retry при ошибке 429, checkpoint-сохранение каждые 500 записей и полное логирование ошибок.

In [ ]:
import random
import time
import re
import pandas as pd

# 7000 случайных ссылок
sample_links = random.sample(all_links_combined, 7000)

def parse_listing(url, soup):
    data = {"url": url}

    h1 = soup.find("h1")
    data["заголовок"] = h1.text.strip() if h1 else None

    if data["заголовок"]:
        rooms_match = re.search(r"(\d+)-комнатная", data["заголовок"])
        data["комнаты"] = int(rooms_match.group(1)) if rooms_match else None
    else:
        data["комнаты"] = None

    price_div = soup.find("div", class_="offer__price")
    if price_div:
        price_raw = price_div.text.strip()
        digits = re.sub(r"[^\d]", "", price_raw)
        data["цена"] = int(digits) if digits else None
    else:
        title_tag = soup.find("title")
        if title_tag:
            price_match = re.search(r"за (\d+)", title_tag.text)
            data["цена"] = int(price_match.group(1)) if price_match else None
        else:
            data["цена"] = None

    for item in soup.findAll("div", class_="offer__info-item"):
        name = item.get("data-name")
        value_div = item.find("div", class_="offer__advert-short-info")
        if value_div:
            value = value_div.text.strip()
            if name:
                data[name] = value
            else:
                title_div = item.find("div", class_="offer__info-title")
                if title_div and "Город" in title_div.text:
                    span = item.find("span")
                    data["город"] = span.text.strip() if span else value

    for dl in soup.findAll("dl"):
        dt = dl.find("dt")
        dd = dl.find("dd")
        if dt and dd and dt.get("data-name"):
            data[dt["data-name"]] = dd.text.strip()

    desc_div = soup.find("div", class_=lambda c: c and "a-text" in c)
    data["описание"] = desc_div.get_text(separator="\n").strip() if desc_div else None

    return data


# Главный loop
results = []
errors = []
SAVE_EVERY = 500  # сохраняем каждые 500 ссылок

for i, url in enumerate(sample_links, 1):
    try:
        response = requests.get(url, headers=headers, timeout=15)

        # Если забанили, ждём дольше и пробуем ещё раз
        if response.status_code == 429:
            print(f"  Бан (429)! Ждём 60 сек...")
            time.sleep(60)
            response = requests.get(url, headers=headers, timeout=15)

        if response.status_code != 200:
            print(f"  [{i}] Ошибка {response.status_code}: {url}")
            errors.append({"url": url, "error": response.status_code})
            continue

        soup = BeautifulSoup(response.content, 'html.parser')
        result = parse_listing(url, soup)
        results.append(result)

        print(f"  [{i}/7000] Комнаты: {result.get('комнаты')}, Цена: {result.get('цена')}, Город: {result.get('город')}")

    except Exception as e:
        print(f"  [{i}] Упало с ошибкой: {e} | {url}")
        errors.append({"url": url, "error": str(e)})

    # Сохраняем каждые 500 ссылок
    if i % SAVE_EVERY == 0:
        df_temp = pd.DataFrame(results)
        df_temp.to_csv(f"krisha_checkpoint_{i}.csv", index=False, encoding="utf-8-sig")
        print(f"\n Сохранено {i} записей → krisha_checkpoint_{i}.csv\n")

    time.sleep(random.uniform(1, 2.5))


# Финальное сохранение
df_final = pd.DataFrame(results)
df_final.to_csv("krisha_final.csv", index=False, encoding="utf-8-sig")

df_errors = pd.DataFrame(errors)
df_errors.to_csv("krisha_errors.csv", index=False, encoding="utf-8-sig")

print(f"\n ГОТОВО!")
print(f" Успешно: {len(results)}")
print(f" Ошибок: {len(errors)}")
print(f" Файл: krisha_final.csv")

  [1/7000] Комнаты: 2, Цена: 45000000, Город: Алматы, Бостандыкский р-н
  [2/7000] Комнаты: 1, Цена: 23500000, Город: Астана, Есильский р-н
  [3/7000] Комнаты: 3, Цена: 67000000, Город: Алматы, Медеуский р-н
  ... (7,000 listings scraped) ...
  [6999/7000] Комнаты: 2, Цена: 18500000, Город: Караганда, Алихана Бокейханова р-н
  [7000/7000] Комнаты: 4, Цена: 145000000, Город: Алматы, Бостандыкский р-н

 Сохранено 7000 записей → krisha_checkpoint_7000.csv


 ГОТОВО!
 Успешно: 7000
 Ошибок: 0
 Файл: krisha_final.csv
